In [ ]:
# RSNA Knee FROZEN + FINE-TUNED RadImageNet ensemble (Kaggle CPU, internet OFF). 2 ResNet50 encoders,
# weighted 0.55 frozen (5-fold head) + 0.45 fine-tuned (single head). gold-58 0.8759 vs frozen 0.8718.
import os, glob, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50
import pydicom, cv2
torch.set_num_threads(os.cpu_count() or 4)
K=18; IMG=384; RES=224; EMB_BS=48; W_FROZEN, W_FT = 0.4, 0.6
LABELS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA',
        'Effusion','Synovitis',"Baker's",'Contusion','Fracture']
dev='cuda' if torch.cuda.is_available() else 'cpu'
print('torch',torch.__version__,'| device',dev)
base=None
for r in sorted(glob.glob('/kaggle/input/*')):
    h=glob.glob(r+'/**/test_series.csv', recursive=True)
    if h: base=os.path.dirname(h[0]); break
print('base',base)
test_series=pd.read_csv(os.path.join(base,'test_series.csv'))
sub=pd.read_csv(os.path.join(base,'sample_submission.csv'))
print('test studies', sub.shape[0])

In [ ]:
class Enc(nn.Module):
    def __init__(s): super().__init__(); s.backbone=nn.Sequential(*list(resnet50(weights=None).children())[:-2])
    def forward(s,x): return s.backbone(x).mean(dim=(2,3))
class MIL(nn.Module):
    def __init__(s,di=2048,d=256,n=12):
        super().__init__(); s.proj=nn.Sequential(nn.Linear(di,d),nn.LayerNorm(d),nn.GELU())
        s.plane=nn.Embedding(3,d); s.fluid=nn.Embedding(2,d)
        s.V=nn.Linear(d,128); s.U=nn.Linear(d,128); s.w=nn.Linear(128,1)
        s.head=nn.Sequential(nn.Dropout(0.3), nn.Linear(d,n))
    def forward(s,x,pid,fb):
        h=s.proj(x)+s.plane(pid)+s.fluid(fb)
        a=torch.softmax(s.w(torch.tanh(s.V(h))*torch.sigmoid(s.U(h))),0)
        return s.head((a*h).sum(0))
fw=glob.glob('/kaggle/input/**/ResNet50_radimagenet.pt', recursive=True)[0]
encF=Enc(); encF.load_state_dict(torch.load(fw,map_location='cpu'),strict=True); encF=encF.eval().to(dev)
ckh=torch.load(glob.glob('/kaggle/input/**/head_weights_radimagenet.pt',recursive=True)[0],map_location='cpu')
headsF=[]
for sd in ckh['heads']:
    m=MIL(2048).to(dev).eval(); m.load_state_dict(sd); headsF.append(m)
ckt=torch.load(glob.glob('/kaggle/input/**/finetune_f1_radimagenet.pt',recursive=True)[0],map_location='cpu')
encT=Enc(); encT.load_state_dict(ckt['enc'],strict=True); encT=encT.eval().to(dev)
headT=MIL(int(ckt.get('di',2048))).to(dev).eval(); headT.load_state_dict(ckt['head'],strict=True)
print('frozen heads',len(headsF),'| ft gold58',ckt.get('gold58'))
PL={'Sagittal':0,'Coronal':1,'Axial':2}; print('ready')

In [ ]:
def sampled(sd,k):
    fs=sorted(glob.glob(sd+'/*.dcm')); n=len(fs)
    if n==0: return []
    if n<=k: return fs
    lo,hi=int(n*.12),int(n*.88)
    if hi<=lo: lo,hi=0,n
    return [fs[int(round(x))] for x in np.linspace(lo,hi-1,k)]
def slc(p):
    try:
        d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
        a=a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
        lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
        return cv2.resize(a,(IMG,IMG),interpolation=cv2.INTER_AREA).astype(np.float32)
    except Exception: return None
@torch.no_grad()
def emb(encf,imgs):
    x=torch.from_numpy(np.stack(imgs))[:,None]; x=x*2.0-1.0; x=x.expand(-1,3,-1,-1).contiguous()
    if x.shape[-1]!=RES: x=F.interpolate(x,size=(RES,RES),mode='bilinear',align_corners=False)
    return encf(x.to(dev)).float()
@torch.no_grad()
def predict_study(study):
    imgs=[];pids=[];fbs=[]
    for _,sr in test_series[test_series['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base,'test_series',study,sr['SeriesInstanceUID'])
        if not os.path.isdir(sd):
            alt=glob.glob(os.path.join(base,'**',sr['SeriesInstanceUID']),recursive=True); sd=alt[0] if alt else None
        if not sd: continue
        pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for p in sampled(sd,K):
            im=slc(p)
            if im is not None: imgs.append(im);pids.append(pid);fbs.append(fb)
    if not imgs: return np.full(12,0.5)
    pid=torch.tensor(pids,device=dev); fb=torch.tensor(fbs,device=dev)
    EF=[];ET=[]
    for i in range(0,len(imgs),EMB_BS):
        b=imgs[i:i+EMB_BS]; EF.append(emb(encF,b)); ET.append(emb(encT,b))
    XF=torch.cat(EF,0); XT=torch.cat(ET,0)
    pF=np.mean([torch.sigmoid(h(XF,pid,fb)).cpu().numpy() for h in headsF],0)
    pT=torch.sigmoid(headT(XT,pid,fb)).cpu().numpy()
    return W_FROZEN*pF + W_FT*pT
print('fns ready')

In [ ]:
ids=list(sub['StudyInstanceUID']); t0=time.time(); out={}
for i,s in enumerate(ids):
    out[s]=predict_study(s)
    if i%100==0: print(f'{i}/{len(ids)} {time.time()-t0:.0f}s',flush=True)
res=pd.DataFrame({'StudyInstanceUID':ids})
for j,l in enumerate(LABELS): res[l]=[out[s][j] for s in ids]
res=res[['StudyInstanceUID']+LABELS]; res.to_csv('submission.csv',index=False)
print('WROTE submission.csv',res.shape,f'{time.time()-t0:.0f}s'); print(res.head())